# `generate_coordinate_grid()`

The grid utility `nematics3d.generate_coordinate_grid()` constructs the coordinates of a target regular grid in the index space of a source array. Use it when you know the source and target shapes and want the target samples to span the complete source-index domain along every axis.

Three facts are important from the beginning:

- `shape_source` and `shape_target` must have the same dimensionality, but the function is not restricted to three dimensions;
- the returned array has shape `(*shape_target, ndim)`, with the final axis storing source-index coordinates;
- both endpoints of every source-index interval are retained, so changing `shape_target` changes the sampling spacing rather than the covered domain.


## Setup

This setup cell imports the package used throughout the tutorial.


In [1]:
import nematics3d as n3d


## Minimal example: resample one axis

A source array of length 5 has index coordinates from 0 through 4. Asking for 4 target samples keeps those endpoints and distributes the target coordinates uniformly between them.


In [2]:
grid = n3d.generate_coordinate_grid((5,), (4,))
print(grid)
print("shape:", grid.shape)


[[0.        ]
 [1.33333333]
 [2.66666667]
 [4.        ]]
shape: (4, 1)


The four rows are the four target samples. The final axis has length one because the source is one-dimensional. Fractional values are source-index coordinates, not interpolated field values; a separate interpolation step would evaluate data at those positions.


## Inputs and outputs

The public signature is:

```python
generate_coordinate_grid(
    shape_source,
    shape_target,
)
```

### `shape_source` and `shape_target`

Both arguments are non-empty sequences of positive integers. Python integers and `NumPy` integer scalars are accepted. Boolean values, floating-point dimensions, zero, negative dimensions, strings, and empty shapes are rejected. The two shapes must contain the same number of dimensions.

| Source shape | Target shape | Dimensionality |
| --- | --- | --- |
| `(5,)` | `(11,)` | 1D |
| `(64, 48)` | `(32, 24)` | 2D |
| `(128, 96, 64)` | `(64, 48, 32)` | 3D |
| `(8, 6, 4, 2)` | `(4, 3, 2, 1)` | 4D |

### Returned coordinate grid

The result is one floating-point `NumPy` array with shape `(*shape_target, ndim)`, where `ndim = len(shape_source)`. If `grid` is the returned array, `grid[i, j, ..., d]` is the source-index coordinate of target point `(i, j, ...)` along source axis `d`.


## Examples


### Two-dimensional downsampling

A source shape `(5, 7)` spans source-index intervals $[0,4]$ and $[0,6]$. A target shape `(3, 4)` samples those intervals with three and four points, respectively.


In [3]:
grid = n3d.generate_coordinate_grid((5, 7), (3, 4))
print(grid)
print("shape:", grid.shape)
print("axis-0 coordinates:", grid[:, 0, 0])
print("axis-1 coordinates:", grid[0, :, 1])
print("last target point:", grid[-1, -1])


[[[0. 0.]
  [0. 2.]
  [0. 4.]
  [0. 6.]]

 [[2. 0.]
  [2. 2.]
  [2. 4.]
  [2. 6.]]

 [[4. 0.]
  [4. 2.]
  [4. 4.]
  [4. 6.]]]
shape: (3, 4, 2)
axis-0 coordinates: [0. 2. 4.]
axis-1 coordinates: [0. 2. 4. 6.]
last target point: [4. 6.]


### Upsampling

Increasing the target size inserts uniformly spaced source-index coordinates between the original integer lattice sites. These fractional coordinates are suitable for a later interpolation step.


In [4]:
grid = n3d.generate_coordinate_grid((3,), (5,))
print(grid[:, 0])


[0.  0.5 1.  1.5 2. ]


### Identical source and target shapes

When the shapes are identical, the target coordinates are exactly the ordinary array indices represented as floating-point values. `GridFieldDataset` uses this case to build its index-space coordinate grid before applying the dataset's grid transform and offset.


In [5]:
grid = n3d.generate_coordinate_grid((2, 3, 2), (2, 3, 2))
print("shape:", grid.shape)
print("grid[1, 2, 1] =", grid[1, 2, 1])


shape: (2, 3, 2, 3)
grid[1, 2, 1] = [1. 2. 1.]


### A target axis of length one

If a target axis contains only one point, that point is placed at source coordinate zero. This follows `numpy.linspace(0, source_size - 1, 1)`. It is therefore an endpoint sample, not a sample of the source-axis center.


In [6]:
grid = n3d.generate_coordinate_grid((9, 5), (1, 3))
print(grid)


[[[0. 0.]
  [0. 2.]
  [0. 4.]]]


## Details

### Coordinate convention

For source-axis length $N_s$ and target-axis length $N_t>1$, target index $i$ is mapped to source coordinate

$$x_i = i\frac{N_s-1}{N_t-1}, \qquad i=0,\ldots,N_t-1.$$

This convention preserves both source endpoints. It differs from cell-centered resampling conventions in which samples represent voxel centers rather than array indices.

### Memory behavior

The returned dense coordinate grid contains `prod(shape_target) * ndim` floating-point values and can therefore be large. During general resampling, the implementation allocates the final grid once and broadcasts one one-dimensional coordinate axis into it at a time, instead of constructing `ndim` full `numpy.meshgrid()` temporaries and stacking them. When source and target shapes are identical, it uses `numpy.indices()` directly.


## Possible issues

### Confusing array indices with physical coordinates

The returned values are source **index-space** coordinates. If lattice spacing, rotation, or offset gives the data a nontrivial physical geometry, apply the corresponding grid transform separately before interpreting the coordinates physically.

### Expecting a center sample when the target length is one

A one-point target axis samples source coordinate zero by design. If a center coordinate is required, construct that coordinate explicitly rather than relying on this function.

### Large dense coordinate grids

A full `(Nx, Ny, Nz, 3)` coordinate array can occupy substantial memory. If a downstream algorithm can operate on separable one-dimensional axes instead of dense point coordinates, constructing those axes directly can be more memory efficient.
